# S5.M6-1 M8 增量（uplift）学习器

**目的**：方法筛选第三阶段（第二批）第 2 步（[DR-M5-014](../../../registry/decision_records/DR-M5-014.yaml) · D-4）。现有模型回答「谁会买」，业务追问的是「谁因为营销才买」。本步在两个公开随机试验上检验：
- 按增量排序（S / T / X / R 四种学习器）是否优于按响应排序（只用处理组拟合的响应模型）；
- 联邦特征（两方全部）相对只用内地特征，能否改善增量排序。

**数据与两方**（沿用 M1 已冻结的字段归属；处理变量与结果只由主动方持有）：
- Hillstrom 邮件营销（64,000 行）：处理 = 收到任一邮件；结果 visit（主）、conversion（次）；
- Criteo Uplift v2.1：伯努利抽样 1 / 14（约 100 万行）；结果 visit（主）、conversion（次）。

**指标**：Qini 系数 = Qini 曲线与随机线之间的面积（横轴人群比例，纵轴每位客户的增量结果），越大越好，0 为随机；另报前 30% 人群的增量（处理组与对照组结果率之差）。

**数据**：公开零售邮件与广告试验，**开发证据**，不能外推银行理财产品的实际增量。

In [1]:
import hashlib
import json
import os
import pathlib
import subprocess
import sys

import numpy as np
import pandas as pd
import yaml

HASH_PREFIX_LEN = 12
REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
CONFIG_PATH = REPO_ROOT / "modules/m5_modeling/configs/s5_m6_uplift.yaml"
m6 = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
COMPONENTS = [REPO_ROOT / "modules/m5_modeling/components" / n for n in ("method_screen.py", "uplift_screen.py")]


def sha(path, n=HASH_PREFIX_LEN):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()[:n]


git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True, cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
for p in [CONFIG_PATH, REPO_ROOT / m6["decision_record"]] + COMPONENTS:
    print(p.relative_to(REPO_ROOT), "sha256:" + sha(p))
print("决策记录:", m6["decision_record"], "| step:", m6["step_id"], "|", m6["purpose"])
print("seed:", m6["seed"], "（基模型无行 / 列采样，拟合确定）| bootstrap:", m6["n_boot"], "次，seed", m6["boot_seed"])
print("git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
RAW = pathlib.Path(os.environ[m6["raw_root_env"]])
RUN = pathlib.Path(os.environ[m6["run_dir_env"]])
launch = {}
print("启动指纹（运行开始时记录；路径只留文件名）：")
for line in (RUN / "fingerprint_launch.txt").read_text(encoding="utf-8").splitlines():
    parts = line.split()
    if len(parts) == 2 and len(parts[0]) == hashlib.sha256().digest_size * 2:
        launch[pathlib.Path(parts[1]).name] = parts[0]
        print("  sha256:" + parts[0][:HASH_PREFIX_LEN], pathlib.Path(parts[1]).name)
    else:
        print(" ", line)
for p in COMPONENTS[1:] + [CONFIG_PATH]:
    assert launch[p.name] == hashlib.sha256(p.read_bytes()).hexdigest(), p.name
print("运行时组件与配置与当前仓库逐字节相同。")

modules/m5_modeling/configs/s5_m6_uplift.yaml sha256:89936dd48a90
registry/decision_records/DR-M5-014.yaml sha256:86afb79851b0
modules/m5_modeling/components/method_screen.py sha256:05e19e025d0b
modules/m5_modeling/components/uplift_screen.py sha256:7309342add70
决策记录: registry/decision_records/DR-M5-014.yaml | step: S5.M6 | 增量学习器能否比「谁会买」的响应模型更好地找出「因营销才买」的客户；联邦特征对增量排序有无帮助
seed: 11 （基模型无行 / 列采样，拟合确定）| bootstrap: 1000 次，seed 20261102
git: 9d1421e（工作区含本步未提交改动）
启动指纹（运行开始时记录；路径只留文件名）：
  launch 2026-11-03T20:41:10+08:00 git 674a299
  sha256:1e465cbb281d DR-M5-014.yaml
  sha256:7309342add70 uplift_screen.py
  sha256:89936dd48a90 s5_m6_uplift.yaml
  sha256:156526f4e5ed test_uplift_screen.py
  sha256:05e19e025d0b method_screen.py
  sha256:e4bfdf6fe3d4 method_screen_data.py
  sha256:5f7a6b1b108c method_screen_cv.py
  sha256:761bae66da90 s5_m1_method_screen.yaml
  sha256:6841f6884602 s5_m3_cv_combo.yaml
  sha256:662388e6c02c run_staged.py
运行时组件与配置与当前仓库逐字节相同。


## 1. 数据规模

各切分的行数、处理组人数与两组结果数（处理占比：Hillstrom 约 2 / 3，Criteo 约 85%）。

In [2]:
from modules.m5_modeling.components import uplift_screen as U

RES = {n: json.loads((RUN / (n + ".json")).read_text(encoding="utf-8")) for n in m6["datasets"]}
rows = []
for n, r in RES.items():
    for o, ro in r["outcomes"].items():
        for part, c in ro["counts"].items():
            rows.append({"dataset": n, "outcome": o, "part": part, **c,
                         "rate_treated": c["events_treated"] / c["treated"],
                         "rate_control": c["events_control"] / (c["rows"] - c["treated"])})
counts = pd.DataFrame(rows)
print("两方特征：", {n: {"内地": r["guest_cols"], "香港": r["host_cols"]} for n, r in RES.items()})
print(counts.round(m6["round_qini"]).to_string(index=False))

两方特征： {'hillstrom': {'内地': ['recency', 'history', 'mens', 'womens'], '香港': ['newbie', 'zip_code_Rural', 'zip_code_Surburban', 'zip_code_Urban', 'channel_Multichannel', 'channel_Phone', 'channel_Web']}, 'criteo': {'内地': ['f0', 'f2', 'f4', 'f6', 'f8', 'f10'], '香港': ['f1', 'f3', 'f5', 'f7', 'f9', 'f11']}}
  dataset    outcome  part   rows  treated  events_treated  events_control  rate_treated  rate_control
hillstrom      visit train  38399    25616            4228            1306       0.16505       0.10217
hillstrom      visit   val  12800     8539            1423             482       0.16665       0.11312
hillstrom      visit  test  12801     8539            1481             474       0.17344       0.11122
hillstrom conversion train  38399    25616             273              73       0.01066       0.00571
hillstrom conversion   val  12800     8539              91              24       0.01066       0.00563
hillstrom conversion  test  12801     8539              92              25    

## 2. 复现

下一格从原始文件重跑 Hillstrom 全部流程（Criteo 约 1 小时，不重跑），要求测试 Qini 与批量运行相同、学习器选择相同。

In [3]:
NAME = m6["repro_dataset"]
spec = m6["datasets"][NAME]
again = U.run_dataset(NAME, U.LOADERS[NAME](RAW / spec["path"], spec), m6, spec["outcomes"])
diff = max(abs(again["outcomes"][o]["test"][k]["qini"]["est"] - RES[NAME]["outcomes"][o]["test"][k]["qini"]["est"])
           for o in spec["outcomes"] for k in RES[NAME]["outcomes"][o]["test"])
same_choice = all(again["outcomes"][o]["choice"] == RES[NAME]["outcomes"][o]["choice"] for o in spec["outcomes"])
print("从原始文件重跑 %s：测试 Qini 最大差 %.2g，学习器选择相同 %s" % (NAME, diff, same_choice))
assert diff <= m6["repro_tol"] and same_choice

[03:56:37] hillstrom · visit · main：配置 {'lr': 0.05, 'depth': 3, 'lam': 1.0, 'rounds': 100}，验证 Qini {'S': 0.00196, 'T': 0.00218, 'X': 0.00242, 'R': 0.00239, 'response': 0.00155} → X，用时 5.3s


[03:56:49] hillstrom · visit · fed：配置 {'lr': 0.05, 'depth': 3, 'lam': 1.0, 'rounds': 300}，验证 Qini {'S': 0.00211, 'T': 0.00248, 'X': 0.00236, 'R': 0.00109, 'response': 0.001} → T，用时 12.7s


[03:57:05] hillstrom · conversion · main：配置 {'lr': 0.05, 'depth': 3, 'lam': 1.0, 'rounds': 100}，验证 Qini {'S': -5e-05, 'T': 0.00035, 'X': 0.00023, 'R': 0.00041, 'response': 0.00045} → R，用时 4.9s


[03:57:13] hillstrom · conversion · fed：配置 {'lr': 0.05, 'depth': 3, 'lam': 1.0, 'rounds': 100}，验证 Qini {'S': -0.00013, 'T': -0.0001, 'X': -4e-05, 'R': 9e-05, 'response': 0.00012} → R，用时 7.3s


从原始文件重跑 hillstrom：测试 Qini 最大差 0，学习器选择相同 True


## 3. 结果

- 每个（数据集，结果，特征集）在验证集上按 Qini 选学习器，测试集评一次；
- 基模型不做行 / 列采样、交叉拟合折固定，拟合确定，区间只来自测试行 bootstrap（1,000 次）；
- 三项预先写定的对比在每个（数据集，结果）内做 Holm。

In [4]:
OUT = REPO_ROOT / m6["results_dir"]
OUT.mkdir(parents=True, exist_ok=True)
val, test, con, cfgs = [], [], [], []
for n, r in RES.items():
    for o, ro in r["outcomes"].items():
        for fs in U.FEATURE_SETS:
            cfgs.append({"dataset": n, "outcome": o, "features": fs, **ro["configs"][fs]["chosen"], "choice": ro["choice"][fs]})
            for k, q in ro["val"][fs].items():
                val.append({"dataset": n, "outcome": o, "features": fs, "method": k, "val_qini": q,
                            "chosen": k == ro["choice"][fs]})
        for key, v in ro["test"].items():
            fs, k = key.split(":")
            test.append({"dataset": n, "outcome": o, "features": fs, "method": k, "chosen": k == ro["choice"][fs],
                         "qini": v["qini"]["est"], "qini_ci_low": v["qini"]["ci_low"], "qini_ci_high": v["qini"]["ci_high"],
                         "uplift_top": v["uplift_top"]["est"], "uplift_ci_low": v["uplift_top"]["ci_low"],
                         "uplift_ci_high": v["uplift_top"]["ci_high"]})
        for c in ro["contrasts"]:
            con.append({"dataset": n, "outcome": o, "contrast": c["label"], "a": c["a"], "b": c["b"], "diff": c["diff"],
                        "ci_low": c["ci_low"], "ci_high": c["ci_high"], "p": c["p"], "p_holm": c["p_holm"]})
val, test, con, cfgs = (pd.DataFrame(x) for x in (val, test, con, cfgs))
counts.to_csv(OUT / "counts.csv", index=False, float_format="%.5f")
cfgs.to_csv(OUT / "configs.csv", index=False)
val.to_csv(OUT / "validation.csv", index=False, float_format="%.6f")
test.to_csv(OUT / "test.csv", index=False, float_format="%.6f")
con.to_csv(OUT / "contrasts.csv", index=False, float_format="%.6f")
R = m6["round_qini"]
print("配置与验证集选定的学习器：")
print(cfgs.to_string(index=False))
print("测试集 Qini 系数与前 %.0f%% 人群增量（95%% 区间，测试行 bootstrap）：" % (m6["top_frac"] * 100))
print(test.round(R).to_string(index=False))
print("预先写定的三项对比（Holm 在每个数据集 × 结果内）：")
print(con.round(R).to_string(index=False))

配置与验证集选定的学习器：
  dataset    outcome features   lr  depth  lam  rounds choice
hillstrom      visit     main 0.05      3  1.0     100      X
hillstrom      visit      fed 0.05      3  1.0     300      T
hillstrom conversion     main 0.05      3  1.0     100      R
hillstrom conversion      fed 0.05      3  1.0     100      R
   criteo      visit     main 0.05      5  1.0     300      S
   criteo      visit      fed 0.05      5  1.0     300      S
   criteo conversion     main 0.05      3  1.0     300      X
   criteo conversion      fed 0.05      3  1.0     300      R
测试集 Qini 系数与前 30% 人群增量（95% 区间，测试行 bootstrap）：
  dataset    outcome features   method  chosen     qini  qini_ci_low  qini_ci_high  uplift_top  uplift_ci_low  uplift_ci_high
hillstrom      visit     main        S   False  0.00367      0.00095       0.00594     0.08826        0.06510         0.10977
hillstrom      visit     main        T   False  0.00213     -0.00056       0.00440     0.07764        0.05413         0.10141
hill

## 4. 读法

**复现**：从原始文件重跑 Hillstrom 全部流程，测试 Qini 与批量运行相同（最大差 0），两个结果、两个特征集的学习器选择相同。

**预先写定的三项对比**（Qini 系数配对差，测试行 bootstrap 1,000 次；Holm 在每个数据集 × 结果内；「优于 / 劣于」= 95% 区间整体在 0 以上 / 以下）：

| 数据集 · 结果（测试事件数） | 选定学习器（内地 / 联邦） | ① 联邦选定 − 内地选定 | ② 联邦选定 − 联邦响应模型 | ③ 内地选定 − 内地响应模型 |
|---|---|---|---|---|
| Hillstrom · visit（1,955） | X / T | −0.0020 [−0.0040, −0.0001]，p 0.038，Holm 0.11 | −0.0009（不显著） | +0.0016（不显著） |
| Hillstrom · conversion（117） | R / R | +0.00036 [+0.000004, +0.00076]，p 0.048，Holm 0.14 | +0.00045（不显著） | +0.00022（不显著） |
| Criteo · visit（9,426） | S / S | +0.00012（不显著） | +0.00023（不显著） | +0.00010（不显著） |
| Criteo · conversion（566） | X / R | −0.00003（不显著） | −0.00019（不显著） | −0.00016（不显著） |

- 12 项对比经 Holm 后**没有一项显著**；名义显著（区间不含 0、Holm 后不显著）2 项，都在 Hillstrom 上，方向相反。
- **按增量排序没有显著优于按响应排序**（② ③ 共 8 项，0 项显著）。Criteo 上响应模型与最好的增量学习器几乎一样（visit 0.0034 对 0.0035 ~ 0.0037；conversion 响应模型 0.00034 反而最高）：这个试验里「会访问 / 会转化的人」与「因广告才访问 / 转化的人」排序高度重合。
- Hillstrom visit 上有一处描述性差别：内地 S / X / R 的测试 Qini 区间都不含 0，内地响应模型的区间含 0；但 ③ 的配对差不显著。
- **联邦特征没有显著改善增量排序**（① 共 4 项，0 项显著）。Hillstrom visit 的名义负值来自选择：联邦一侧验证集上 T（0.00248）略高于 S（0.00211）而选了 T，T 在测试集上是联邦五个模型里最低的（0.0014），联邦 S 的测试 Qini（0.0038）其实是全部十个里最高的。这个量级上验证集 Qini 分不开学习器，与 S5.M1 单一验证集分不开 0.005 量级 AUROC 差的情况相同。
- conversion 事件少（测试集 Hillstrom 117 个、Criteo 566 个），Hillstrom 上十个模型的测试 Qini 区间全部含 0，不支持任何结论。

**对报告的意义**：M8 不是提高 AUROC 的方法，它回答「营销带来多少增量」。在这两个公开随机试验上，增量学习器没有显著优于响应模型，联邦特征也没有显著改善增量排序。按「在个别数据集上有明显增益就写进报告」的标准，M8 本步**没有明显增益**，报告写为「已做、未见增益」，并写明三个原因：事件少、增量与响应排序高度重合、验证集选学习器不稳。

**边界**：公开零售邮件与广告随机试验，开发证据，不能外推银行理财产品的实际增量，不换算成汇丰效果；明文代理，未在真实 SGB 上复核；单次确定性拟合，区间只含测试行抽样；Criteo 只用 1 / 14 伯努利抽样。技术侧自评，未经法务复核（W-001）。